In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import shutil
import re

import numpy as np
import pandas as pd
import cv2
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [6]:
from pathlib import Path

# Find the attached UnifiedPerception dataset/project
matches = []

for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_dir() and p.name.lower() == "unifiedperception":
                matches.append(p)

print("UnifiedPerception locations found:")
for p in matches:
    print(" ", p)

if not matches:
    print("\n❌ UnifiedPerception folder was not found.")
    print("\nKaggle inputs currently available:")
    for p in sorted(Path("/kaggle/input").iterdir()):
        print(" ", p)

    raise FileNotFoundError(
        "UnifiedPerception input location could not be found."
    )

# Use the first discovered location
PROJECT = matches[0]

print("\n✅ PROJECT =", PROJECT)

# Check that this looks like the correct project
important = [
    PROJECT / "unified_perception",
    PROJECT / "third_party",
]

for p in important:
    print(
        ("✅" if p.exists() else "⚠️"),
        p
    )

print("\n🎯 Stage 2 project path is ready.")

UnifiedPerception locations found:
  /kaggle/input/datasets/saanvishetty0804/unifiedperception
  /kaggle/input/datasets/saanvishetty0804/unifiedperception/UnifiedPerception

✅ PROJECT = /kaggle/input/datasets/saanvishetty0804/unifiedperception
⚠️ /kaggle/input/datasets/saanvishetty0804/unifiedperception/unified_perception
⚠️ /kaggle/input/datasets/saanvishetty0804/unifiedperception/third_party

🎯 Stage 2 project path is ready.


In [8]:
from pathlib import Path
import sys

BASE = Path("/kaggle/input/datasets/saanvishetty0804/unifiedperception")

print("Contents of:", BASE)
for p in sorted(BASE.iterdir()):
    print(" ", "DIR " if p.is_dir() else "FILE", p.name)

# Find the actual directory containing the required project folders
candidates = [
    BASE,
    BASE / "UnifiedPerception",
]

PROJECT = None

for candidate in candidates:
    if (
        candidate.exists()
        and (candidate / "unified_perception").exists()
        and (candidate / "third_party").exists()
    ):
        PROJECT = candidate
        break

if PROJECT is None:
    raise FileNotFoundError(
        "❌ Could not find a UnifiedPerception directory containing "
        "'unified_perception' and 'third_party'."
    )

print("\n✅ Correct PROJECT:")
print(PROJECT)

# Add the correct paths
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / "third_party" / "training_package"))
sys.path.insert(0, str(PROJECT / "third_party" / "training_package" / "scripts"))

from unified_perception.backbone import extract_features

MODEL_NAME = "vits14"
IMAGE_SIZE = 640
HIDDEN_LAYERS = [3, 6, 9, 12]

print("\nModel:", MODEL_NAME)
print("Image size:", IMAGE_SIZE)
print("Hidden layers:", HIDDEN_LAYERS)

print("\n🎯 DINOv2 feature extractor imported successfully.")
print("✅ Stage 2 environment is ready.")

Contents of: /kaggle/input/datasets/saanvishetty0804/unifiedperception
  DIR  UnifiedPerception

✅ Correct PROJECT:
/kaggle/input/datasets/saanvishetty0804/unifiedperception/UnifiedPerception

Model: vits14
Image size: 640
Hidden layers: [3, 6, 9, 12]

🎯 DINOv2 feature extractor imported successfully.
✅ Stage 2 environment is ready.


In [14]:
# Install the missing video/image dependency used by UnifiedPerception
!pip install -q decord

# Verify
import decord

print("decord version:", decord.__version__)
print("✅ decord installed successfully.")
print("Now rerun the WoodScape DINOv2 smoke-test block.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 99.2 MB/s eta 0:00:00:00:01:01
decord version: 0.6.0
✅ decord installed successfully.
Now rerun the WoodScape DINOv2 smoke-test block.


In [15]:
# ============================================================
# FINAL DINOv2 + WOODSCAPE SMOKE TEST
# Fixes Hugging Face cache path at the library-constant level
# ============================================================

import os
import sys
import time
from pathlib import Path

import torch

# ------------------------------------------------------------
# 1. Writable Kaggle cache
# ------------------------------------------------------------

HF_CACHE = Path("/kaggle/working/huggingface_cache")
HF_HUB_CACHE = HF_CACHE / "hub"
TRANSFORMERS_CACHE = HF_CACHE / "transformers"

HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

# Environment variables
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)
os.environ["XDG_CACHE_HOME"] = str(HF_CACHE)

# ------------------------------------------------------------
# 2. IMPORTANT:
# Patch already-imported Hugging Face constants too
# ------------------------------------------------------------

import huggingface_hub.constants as hf_constants

hf_constants.HF_HOME = str(HF_CACHE)
hf_constants.HF_HUB_CACHE = str(HF_HUB_CACHE)

# Patch transformers' cached hub location if present
try:
    import transformers.utils.hub as transformers_hub

    if hasattr(transformers_hub, "HF_HUB_CACHE"):
        transformers_hub.HF_HUB_CACHE = str(HF_HUB_CACHE)

except Exception:
    pass

print("=" * 70)
print("CACHE CONFIGURATION")
print("=" * 70)

print("HF_HOME       :", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE  :", os.environ.get("HF_HUB_CACHE"))
print("TRANSFORMERS  :", os.environ.get("TRANSFORMERS_CACHE"))
print("Writable cache:", HF_CACHE)

# ------------------------------------------------------------
# 3. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/saanvishetty0804/car-nvidia"
)

PROJECT = Path(
    "/kaggle/input/datasets/saanvishetty0804/"
    "unifiedperception/UnifiedPerception"
)

CURRENT_DIR = DATA_ROOT / "rgb_images-001" / "rgb_images"

assert DATA_ROOT.exists()
assert PROJECT.exists()
assert CURRENT_DIR.exists()

# ------------------------------------------------------------
# 4. UnifiedPerception Python path
# ------------------------------------------------------------

sys.path.insert(0, str(PROJECT))
sys.path.insert(
    0,
    str(PROJECT / "third_party" / "training_package")
)
sys.path.insert(
    0,
    str(PROJECT / "third_party" / "training_package" / "scripts")
)

# ------------------------------------------------------------
# 5. Import UnifiedPerception backbone
# ------------------------------------------------------------

import unified_perception.backbone as backbone

# ------------------------------------------------------------
# 6. Patch the backbone loader
# ------------------------------------------------------------

from transformers import AutoModel


def patched_load_backbone(model_name, device):

    model_names = {
        "vits14": "facebook/dinov2-small",
        "vitb14": "facebook/dinov2-base",
        "vitl14": "facebook/dinov2-large",
        "vitg14": "facebook/dinov2-giant",
    }

    if model_name not in model_names:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    hf_id = model_names[model_name]

    print(
        f"\n[PATCHED] Loading {model_name} "
        f"({hf_id}) on {device}"
    )

    print(
        "[PATCHED] Using cache:",
        HF_HUB_CACHE
    )

    model = AutoModel.from_pretrained(
        hf_id,
        cache_dir=str(HF_HUB_CACHE),
    )

    model = model.to(device).eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    return model


backbone._load_backbone = patched_load_backbone

extract_features = backbone.extract_features

# ------------------------------------------------------------
# 7. GPU
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), "CUDA is not available."

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 8. Select REAL WoodScape image
# ------------------------------------------------------------

images = sorted(CURRENT_DIR.glob("*.png"))

assert len(images) > 0

sample_path = images[0]

print("\nWoodScape sample:")
print(sample_path)

# ------------------------------------------------------------
# 9. DINOv2 configuration
# ------------------------------------------------------------

MODEL_NAME = "vits14"
IMAGE_SIZE = 640
HIDDEN_LAYERS = [3, 6, 9, 12]

# ------------------------------------------------------------
# 10. Run
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING DINOv2 ON REAL WOODSCAPE IMAGE")
print("=" * 70)

start = time.perf_counter()

features = extract_features(
    str(sample_path),
    num_frames=1,
    model_name=MODEL_NAME,
    device="cuda",
    image_size=IMAGE_SIZE,
    hidden_layer_indices=HIDDEN_LAYERS,
    keep_frames=False,
)

runtime = time.perf_counter() - start

# ------------------------------------------------------------
# 11. Validate
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DINOv2 RESULTS")
print("=" * 70)

print("Runtime:", round(runtime, 3), "seconds")
print("CLS:", tuple(features.cls.shape))
print(
    "Patch tokens:",
    tuple(features.patch_tokens.shape)
)

print(
    "Hidden layers:",
    sorted(features.hidden_states.keys())
)

for layer in HIDDEN_LAYERS:
    print(
        f"Layer {layer}:",
        tuple(features.hidden_states[layer].shape)
    )

# Expected ViT-S/14 @ 640:
# 45 x 45 = 2025 patch tokens
# + CLS = 2026 tokens

assert tuple(features.cls.shape) == (1, 384)

assert tuple(
    features.patch_tokens.shape
) == (1, 2025, 384)

for layer in HIDDEN_LAYERS:
    assert tuple(
        features.hidden_states[layer].shape
    ) == (1, 2026, 384)

# ------------------------------------------------------------
# 12. SUCCESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 WOODSCAPE DINOv2 SMOKE TEST PASSED")
print("=" * 70)

print("Real WoodScape frame:", sample_path.name)
print("DINOv2:", MODEL_NAME)
print("Patch tokens: 2025")
print("Feature dimension: 384")
print("GPU:", torch.cuda.get_device_name(0))

print("\n✅ Stage-2 DINOv2 feature extraction is ready.")

CACHE CONFIGURATION
HF_HOME       : /kaggle/working/huggingface_cache
HF_HUB_CACHE  : /kaggle/working/huggingface_cache/hub
TRANSFORMERS  : /kaggle/working/huggingface_cache/transformers
Writable cache: /kaggle/working/huggingface_cache

GPU CHECK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4

WoodScape sample:
/kaggle/input/datasets/saanvishetty0804/car-nvidia/rgb_images-001/rgb_images/00000_FV.png

RUNNING DINOv2 ON REAL WOODSCAPE IMAGE

[PATCHED] Loading vits14 (facebook/dinov2-small) on cuda
[PATCHED] Using cache: /kaggle/working/huggingface_cache/hub


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DECORDError: [05:23:01] /github/workspace/src/video/video_reader.cc:270: [/kaggle/input/datasets/saanvishetty0804/car-nvidia/rgb_images-001/rgb_images/00000_FV.png] Failed to measure duration/frame-count due to broken metadata.

In [18]:
# ============================================================
# FINAL WOODSCAPE DINOv2 640x640 SMOKE TEST
# Manually resize image; disable processor resizing
# ============================================================

import time
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/saanvishetty0804/car-nvidia"
)

CURRENT_DIR = DATA_ROOT / "rgb_images-001" / "rgb_images"

sample_path = sorted(CURRENT_DIR.glob("*.png"))[0]

MODEL_ID = "facebook/dinov2-small"
IMAGE_SIZE = 640

HF_CACHE = Path("/kaggle/working/huggingface_cache")

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

print("=" * 70)
print("WOODSCAPE DINOv2 — FINAL 640x640 SMOKE TEST")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Load WoodScape image
# ------------------------------------------------------------

image = Image.open(sample_path).convert("RGB")

print("\nImage:", sample_path.name)
print("Original size:", image.size)

# ------------------------------------------------------------
# IMPORTANT:
# Manually resize to exactly 640x640
# ------------------------------------------------------------

image_640 = image.resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    Image.Resampling.BICUBIC
)

print("Resized image:", image_640.size)

assert image_640.size == (640, 640)

# ------------------------------------------------------------
# Load processor
# ------------------------------------------------------------

processor = AutoImageProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
    use_fast=False,
)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

print("\nLoading DINOv2 model...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    cache_dir=str(HF_CACHE),
    output_hidden_states=True,
)

model = model.to("cuda").eval()

for p in model.parameters():
    p.requires_grad = False

print("✅ DINOv2 model loaded.")

# ------------------------------------------------------------
# Process WITHOUT resizing
# ------------------------------------------------------------

inputs = processor(
    images=image_640,
    return_tensors="pt",
    do_resize=False,
    do_center_crop=False,
)

print("\nProcessor output BEFORE GPU:")
print(tuple(inputs["pixel_values"].shape))

assert tuple(
    inputs["pixel_values"].shape
) == (1, 3, 640, 640), (
    "Processor changed the image size unexpectedly."
)

inputs = {
    key: value.to("cuda")
    for key, value in inputs.items()
}

print("Processor output ON GPU:")
print(tuple(inputs["pixel_values"].shape))

# ------------------------------------------------------------
# Run DINOv2
# ------------------------------------------------------------

print("\nRunning DINOv2...")

torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    outputs = model(**inputs)

torch.cuda.synchronize()

runtime = time.perf_counter() - start

# ------------------------------------------------------------
# Extract tokens
# ------------------------------------------------------------

last_hidden = outputs.last_hidden_state

cls_token = last_hidden[:, 0]
patch_tokens = last_hidden[:, 1:]

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DINOv2 RESULTS")
print("=" * 70)

print("Runtime:", round(runtime, 3), "seconds")
print("Full token shape:", tuple(last_hidden.shape))
print("CLS shape:", tuple(cls_token.shape))
print("Patch-token shape:", tuple(patch_tokens.shape))
print("Hidden-state count:", len(outputs.hidden_states))

# ------------------------------------------------------------
# Expected:
#
# 640x640
# Patch size = 14
# DINOv2 interpolates positional embeddings
# 45 x 45 = 2025 patches
# + 1 CLS = 2026 tokens
# ------------------------------------------------------------

expected_patches = 45 * 45

assert cls_token.shape == (1, 384)

assert patch_tokens.shape == (
    1,
    expected_patches,
    384,
), (
    f"Expected (1, {expected_patches}, 384), "
    f"got {tuple(patch_tokens.shape)}"
)

assert last_hidden.shape == (
    1,
    expected_patches + 1,
    384,
)

# ------------------------------------------------------------
# SUCCESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 WOODSCAPE DINOv2 640x640 SMOKE TEST PASSED")
print("=" * 70)

print("Real WoodScape:", sample_path.name)
print("Input:", "640 x 640")
print("Patch grid:", "45 x 45")
print("Patch tokens:", 2025)
print("Feature dimension:", 384)
print("Total tokens:", 2026)
print("GPU:", torch.cuda.get_device_name(0))

print("\n✅ DINOv2 is ready for the Stage-2 temporal experiment.")

WOODSCAPE DINOv2 — FINAL 640x640 SMOKE TEST
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4

Image: 00000_FV.png
Original size: (1280, 966)
Resized image: (640, 640)

Loading DINOv2 model...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

✅ DINOv2 model loaded.

Processor output BEFORE GPU:
(1, 3, 640, 640)
Processor output ON GPU:
(1, 3, 640, 640)

Running DINOv2...

DINOv2 RESULTS
Runtime: 0.109 seconds
Full token shape: (1, 2026, 384)
CLS shape: (1, 384)
Patch-token shape: (1, 2025, 384)
Hidden-state count: 13

🎉 WOODSCAPE DINOv2 640x640 SMOKE TEST PASSED
Real WoodScape: 00000_FV.png
Input: 640 x 640
Patch grid: 45 x 45
Patch tokens: 2025
Feature dimension: 384
Total tokens: 2026
GPU: Tesla T4

✅ DINOv2 is ready for the Stage-2 temporal experiment.


In [22]:
# ============================================================
# FIXED WOODSCAPE CURRENT / PREVIOUS PAIRING
# Current:  00000_FV.png
# Previous: 00000_FV_prev.png
# ============================================================

from pathlib import Path
import re

DATA_ROOT = Path(
    "/kaggle/input/datasets/saanvishetty0804/car-nvidia"
)

CURRENT_DIR = DATA_ROOT / "rgb_images-001" / "rgb_images"
PREVIOUS_DIR = DATA_ROOT / "previous_images-001" / "previous_images"

CAMERAS = ["FV", "RV", "MVL", "MVR"]

assert CURRENT_DIR.exists(), f"Current directory missing: {CURRENT_DIR}"
assert PREVIOUS_DIR.exists(), f"Previous directory missing: {PREVIOUS_DIR}"

# ------------------------------------------------------------
# Robust filename parser
# ------------------------------------------------------------

def parse_filename(path):
    """
    Handles:
        00000_FV.png
        00000_FV_prev.png

    Returns:
        (frame_id, camera)
    """

    name = path.name

    match = re.match(
        r"^(\d+)_([A-Za-z0-9]+)(?:_prev)?\.png$",
        name
    )

    if match is None:
        return None, None

    frame_id = int(match.group(1))
    camera = match.group(2)

    return frame_id, camera


# ------------------------------------------------------------
# Build previous-frame lookup
# ------------------------------------------------------------

previous_files = sorted(
    PREVIOUS_DIR.glob("*.png")
)

previous_lookup = {}

for path in previous_files:

    frame_id, camera = parse_filename(path)

    if (
        frame_id is not None
        and camera in CAMERAS
    ):
        previous_lookup[
            (camera, frame_id)
        ] = path


# ------------------------------------------------------------
# Match current frames
# ------------------------------------------------------------

current_files = sorted(
    CURRENT_DIR.glob("*.png")
)

pairs = {
    camera: []
    for camera in CAMERAS
}

for current_path in current_files:

    frame_id, camera = parse_filename(
        current_path
    )

    if (
        frame_id is None
        or camera not in CAMERAS
    ):
        continue

    previous_path = previous_lookup.get(
        (camera, frame_id)
    )

    if previous_path is not None:

        pairs[camera].append({
            "camera": camera,
            "frame_id": frame_id,
            "current": current_path,
            "previous": previous_path,
        })


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

for camera in CAMERAS:

    pairs[camera] = sorted(
        pairs[camera],
        key=lambda x: x["frame_id"]
    )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 70)
print("WOODSCAPE CURRENT / PREVIOUS PAIRS")
print("=" * 70)

for camera in CAMERAS:

    print(
        f"{camera}: {len(pairs[camera])} pairs"
    )


# ------------------------------------------------------------
# Show examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PAIRING EXAMPLES")
print("=" * 70)

for camera in CAMERAS:

    if pairs[camera]:

        example = pairs[camera][0]

        print(f"\n{camera}:")
        print(
            "Current :",
            example["current"].name
        )
        print(
            "Previous:",
            example["previous"].name
        )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert all(
    len(pairs[camera]) > 0
    for camera in CAMERAS
), "❌ At least one camera still has zero pairs."

print("\n" + "=" * 70)
print("✅ WOODSCAPE PAIRING PASSED")
print("=" * 70)

print(
    "Current and previous frames are correctly matched "
    "by frame ID + camera."
)

WOODSCAPE CURRENT / PREVIOUS PAIRS
FV: 2037 pairs
RV: 1986 pairs
MVL: 2066 pairs
MVR: 2145 pairs

PAIRING EXAMPLES

FV:
Current : 00000_FV.png
Previous: 00000_FV_prev.png

RV:
Current : 00028_RV.png
Previous: 00028_RV_prev.png

MVL:
Current : 00140_MVL.png
Previous: 00140_MVL_prev.png

MVR:
Current : 00044_MVR.png
Previous: 00044_MVR_prev.png

✅ WOODSCAPE PAIRING PASSED
Current and previous frames are correctly matched by frame ID + camera.


In [23]:
# ============================================================
# STAGE 2 — DINOv2 TEMPORAL TOKEN REDUNDANCY EXPERIMENT
#
# Uses:
#   current WoodScape frame  <-> previous WoodScape frame
#   DINOv2 ViT-S/14
#   640x640 input
#   2025 patch tokens x 384 dimensions
#
# NOTE:
# This measures temporal redundancy.
# It does NOT yet measure actual speedup.
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

# ------------------------------------------------------------
# 1. Check that the successful smoke-test objects exist
# ------------------------------------------------------------

assert "model" in globals(), (
    "DINOv2 model is missing. Run the successful "
    "640x640 smoke test first."
)

assert "processor" in globals(), (
    "DINOv2 processor is missing. Run the successful "
    "640x640 smoke test first."
)

assert "pairs" in globals(), (
    "WoodScape pairs are missing. Run the pairing block first."
)

assert torch.cuda.is_available()

print("=" * 70)
print("STAGE 2 — TEMPORAL DINOv2 TOKEN REDUNDANCY")
print("=" * 70)

print("GPU:", torch.cuda.get_device_name(0))
print("Model: DINOv2 ViT-S/14")
print("Input: 640 x 640")
print("Patch tokens: 2025")
print("Feature dimension: 384")

# ------------------------------------------------------------
# 2. Experiment settings
# ------------------------------------------------------------

CAMERAS = ["FV", "RV", "MVL", "MVR"]

# Start with 8 frame pairs per camera.
# This is intentionally a small pilot experiment.
PAIRS_PER_CAMERA = 8

# Similarity thresholds.
COSINE_THRESHOLDS = [
    0.90,
    0.95,
    0.98,
    0.99,
]

print("\nPairs per camera:", PAIRS_PER_CAMERA)
print("Cosine thresholds:", COSINE_THRESHOLDS)

# ------------------------------------------------------------
# 3. Function to extract DINOv2 patch tokens
# ------------------------------------------------------------

def get_patch_tokens(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    # EXACT same resizing used in the
    # successful 640x640 smoke test.
    image = image.resize(
        (640, 640),
        Image.Resampling.BICUBIC
    )

    # Prevent the processor from resizing it back to 224x224.
    inputs = processor(
        images=image,
        return_tensors="pt",
        do_resize=False,
        do_center_crop=False,
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # Verify input resolution.
    assert tuple(
        inputs["pixel_values"].shape
    ) == (1, 3, 640, 640)

    with torch.inference_mode():

        outputs = model(
            **inputs
        )

    # Remove CLS token.
    #
    # Full output:
    #     1 x 2026 x 384
    #
    # Patch tokens:
    #     1 x 2025 x 384
    #
    patch_tokens = outputs.last_hidden_state[
        :, 1:, :
    ]

    assert tuple(
        patch_tokens.shape
    ) == (1, 2025, 384)

    return patch_tokens[0]


# ------------------------------------------------------------
# 4. Run experiment
# ------------------------------------------------------------

rows = []

experiment_start = time.perf_counter()

for camera in CAMERAS:

    print("\n" + "=" * 70)
    print("CAMERA:", camera)
    print("=" * 70)

    selected_pairs = pairs[camera][
        :PAIRS_PER_CAMERA
    ]

    for index, item in enumerate(
        selected_pairs,
        start=1
    ):

        print(
            f"\n[{camera} "
            f"{index}/{len(selected_pairs)}] "
            f"Frame {item['frame_id']}"
        )

        # ----------------------------------------------------
        # Current frame
        # ----------------------------------------------------

        current_tokens = get_patch_tokens(
            item["current"]
        )

        # ----------------------------------------------------
        # Previous frame
        # ----------------------------------------------------

        previous_tokens = get_patch_tokens(
            item["previous"]
        )

        # ----------------------------------------------------
        # Corresponding patch-token cosine similarity
        # ----------------------------------------------------

        cosine = F.cosine_similarity(
            current_tokens,
            previous_tokens,
            dim=-1
        )

        cosine_np = (
            cosine
            .detach()
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Per-token L2 difference
        # ----------------------------------------------------

        l2_difference = torch.norm(
            current_tokens -
            previous_tokens,
            dim=-1
        )

        current_norm = torch.norm(
            current_tokens,
            dim=-1
        ).clamp(
            min=1e-8
        )

        relative_l2 = (
            l2_difference /
            current_norm
        )

        relative_l2_np = (
            relative_l2
            .detach()
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Statistics for each threshold
        # ----------------------------------------------------

        for threshold in COSINE_THRESHOLDS:

            stable_mask = (
                cosine_np >= threshold
            )

            stable_count = int(
                stable_mask.sum()
            )

            stable_percent = (
                stable_count /
                2025 *
                100.0
            )

            rows.append({

                "camera":
                    camera,

                "frame_id":
                    item["frame_id"],

                "cosine_threshold":
                    threshold,

                "stable_tokens":
                    stable_count,

                "total_tokens":
                    2025,

                "stable_percent":
                    stable_percent,

                "mean_cosine":
                    float(
                        cosine_np.mean()
                    ),

                "median_cosine":
                    float(
                        np.median(
                            cosine_np
                        )
                    ),

                "min_cosine":
                    float(
                        cosine_np.min()
                    ),

                "mean_relative_l2":
                    float(
                        relative_l2_np.mean()
                    ),

                "median_relative_l2":
                    float(
                        np.median(
                            relative_l2_np
                        )
                    ),
            })

        # ----------------------------------------------------
        # Console summary for this frame
        # ----------------------------------------------------

        print(
            "    Mean cosine:",
            round(
                float(cosine_np.mean()),
                4
            )
        )

        print(
            "    Median cosine:",
            round(
                float(
                    np.median(cosine_np)
                ),
                4
            )
        )

        print(
            "    Tokens >= 0.90:",
            round(
                float(
                    (
                        cosine_np >= 0.90
                    ).mean() * 100
                ),
                2
            ),
            "%"
        )

        print(
            "    Tokens >= 0.95:",
            round(
                float(
                    (
                        cosine_np >= 0.95
                    ).mean() * 100
                ),
                2
            ),
            "%"
        )

        print(
            "    Tokens >= 0.98:",
            round(
                float(
                    (
                        cosine_np >= 0.98
                    ).mean() * 100
                ),
                2
            ),
            "%"
        )

# ------------------------------------------------------------
# 5. Create raw results
# ------------------------------------------------------------

token_similarity_df = pd.DataFrame(
    rows
)

# ------------------------------------------------------------
# 6. Aggregate by camera + threshold
# ------------------------------------------------------------

summary = (
    token_similarity_df
    .groupby(
        [
            "camera",
            "cosine_threshold"
        ]
    )
    .agg(

        stable_percent=(
            "stable_percent",
            "mean"
        ),

        stable_std=(
            "stable_percent",
            "std"
        ),

        mean_cosine=(
            "mean_cosine",
            "mean"
        ),

        median_cosine=(
            "median_cosine",
            "mean"
        ),

        mean_relative_l2=(
            "mean_relative_l2",
            "mean"
        ),

        median_relative_l2=(
            "median_relative_l2",
            "mean"
        ),
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

experiment_runtime = (
    time.perf_counter()
    - experiment_start
)

print("\n" + "=" * 70)
print("TEMPORAL TOKEN REDUNDANCY SUMMARY")
print("=" * 70)

display(
    summary.round(4)
)

# ------------------------------------------------------------
# 8. Overall summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL EXPERIMENT")
print("=" * 70)

print(
    "Cameras:",
    len(CAMERAS)
)

print(
    "Pairs per camera:",
    PAIRS_PER_CAMERA
)

print(
    "Total frame pairs:",
    PAIRS_PER_CAMERA *
    len(CAMERAS)
)

print(
    "Patch tokens per frame:",
    2025
)

print(
    "Total comparisons:",
    PAIRS_PER_CAMERA *
    len(CAMERAS) *
    2025
)

print(
    "Runtime:",
    round(
        experiment_runtime,
        2
    ),
    "seconds"
)

print("\n" + "=" * 70)
print("🎉 STAGE 2 TEMPORAL TOKEN EXPERIMENT COMPLETED")
print("=" * 70)

print(
    "\nImportant:"
)

print(
    "stable_percent = percentage of current/previous "
    "patch tokens whose cosine similarity exceeds "
    "the selected threshold."
)

print(
    "\nThis is evidence of temporal redundancy, "
    "NOT measured DINOv2 speedup."
)

STAGE 2 — TEMPORAL DINOv2 TOKEN REDUNDANCY
GPU: Tesla T4
Model: DINOv2 ViT-S/14
Input: 640 x 640
Patch tokens: 2025
Feature dimension: 384

Pairs per camera: 8
Cosine thresholds: [0.9, 0.95, 0.98, 0.99]

CAMERA: FV

[FV 1/8] Frame 0
    Mean cosine: 0.8919
    Median cosine: 0.951
    Tokens >= 0.90: 68.0 %
    Tokens >= 0.95: 50.42 %
    Tokens >= 0.98: 25.68 %

[FV 2/8] Frame 1
    Mean cosine: 0.863
    Median cosine: 0.9361
    Tokens >= 0.90: 61.43 %
    Tokens >= 0.95: 43.31 %
    Tokens >= 0.98: 20.79 %

[FV 3/8] Frame 2
    Mean cosine: 0.9164
    Median cosine: 0.9683
    Tokens >= 0.90: 75.31 %
    Tokens >= 0.95: 60.74 %
    Tokens >= 0.98: 37.33 %

[FV 4/8] Frame 3
    Mean cosine: 0.9115
    Median cosine: 0.966
    Tokens >= 0.90: 74.17 %
    Tokens >= 0.95: 58.81 %
    Tokens >= 0.98: 34.47 %

[FV 5/8] Frame 4
    Mean cosine: 0.9003
    Median cosine: 0.9629
    Tokens >= 0.90: 71.16 %
    Tokens >= 0.95: 57.09 %
    Tokens >= 0.98: 33.53 %

[FV 6/8] Frame 5
    Mean co

,camera,cosine_threshold,stable_percent,stable_std,mean_cosine,median_cosine,mean_relative_l2,median_relative_l2
0,FV,0.90,68.2840,6.0838,0.8901,0.9535,0.3883,0.3029
1,FV,0.95,52.5926,6.6813,0.8901,0.9535,0.3883,0.3029
2,FV,0.98,29.5679,5.8247,0.8901,0.9535,0.3883,0.3029
3,FV,0.99,13.4259,3.8174,0.8901,0.9535,0.3883,0.3029
4,MVL,0.90,53.7963,6.1598,0.8162,0.9110,0.5217,0.4215
5,MVL,0.95,36.0864,6.0096,0.8162,0.9110,0.5217,0.4215
6,MVL,0.98,11.2222,5.7960,0.8162,0.9110,0.5217,0.4215
7,MVL,0.99,1.8951,2.1051,0.8162,0.9110,0.5217,0.4215
8,MVR,0.90,70.9074,6.8614,0.8982,0.9598,0.3700,0.2826
9,MVR,0.95,55.9444,6.3009,0.8982,0.9598,0.3700,0.2826



OVERALL EXPERIMENT
Cameras: 4
Pairs per camera: 8
Total frame pairs: 32
Patch tokens per frame: 2025
Total comparisons: 64800
Runtime: 7.12 seconds

🎉 STAGE 2 TEMPORAL TOKEN EXPERIMENT COMPLETED

Important:
stable_percent = percentage of current/previous patch tokens whose cosine similarity exceeds the selected threshold.

This is evidence of temporal redundancy, NOT measured DINOv2 speedup.


In [24]:
# ============================================================
# SAVE STAGE-2 RESULTS TO KAGGLE WORKING
# ============================================================

from pathlib import Path
import json
import pandas as pd

RESULTS = Path("/kaggle/working/woodscape_results_clean")
RESULTS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Save raw token-similarity results
# ------------------------------------------------------------

raw_path = RESULTS / "stage2_temporal_token_raw.csv"

token_similarity_df.to_csv(
    raw_path,
    index=False
)

# ------------------------------------------------------------
# 2. Save aggregated summary
# ------------------------------------------------------------

summary_path = RESULTS / "stage2_temporal_token_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# 3. Save experiment configuration + summary as JSON
# ------------------------------------------------------------

json_path = RESULTS / "stage2_temporal_token_summary.json"

stage2_info = {
    "stage": "Stage 2 - Temporal DINOv2 Token Redundancy",

    "dataset": "WoodScape",

    "cameras": CAMERAS,

    "model": "DINOv2 ViT-S/14",

    "model_id": "facebook/dinov2-small",

    "input_resolution": "640x640",

    "patch_grid": "45x45",

    "patch_tokens": 2025,

    "feature_dimension": 384,

    "pairs_per_camera": PAIRS_PER_CAMERA,

    "cosine_thresholds": COSINE_THRESHOLDS,

    "total_pairs": (
        PAIRS_PER_CAMERA * len(CAMERAS)
    ),

    "interpretation": {
        "stable_percent":
            "Percentage of corresponding patch tokens "
            "with cosine similarity above the selected threshold.",

        "actual_compute_reduction":
            False,

        "actual_speedup_measured":
            False,

        "purpose":
            "Measure temporal redundancy as evidence "
            "for potential token reuse."
    },

    "summary":
        summary.to_dict(
            orient="records"
        )
}

with open(
    json_path,
    "w"
) as f:

    json.dump(
        stage2_info,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 4. Save a simple text report
# ------------------------------------------------------------

report_path = RESULTS / "stage2_report.txt"

with open(
    report_path,
    "w"
) as f:

    f.write(
        "STAGE 2 — DINOv2 TEMPORAL TOKEN REDUNDANCY\n"
    )

    f.write(
        "=" * 60 + "\n\n"
    )

    f.write(
        "Dataset: WoodScape\n"
    )

    f.write(
        "Model: DINOv2 ViT-S/14\n"
    )

    f.write(
        "Input resolution: 640x640\n"
    )

    f.write(
        "Patch tokens per frame: 2025\n"
    )

    f.write(
        "Feature dimension: 384\n"
    )

    f.write(
        f"Pairs per camera: {PAIRS_PER_CAMERA}\n"
    )

    f.write(
        f"Total pairs: "
        f"{PAIRS_PER_CAMERA * len(CAMERAS)}\n\n"
    )

    f.write(
        "IMPORTANT:\n"
    )

    f.write(
        "This experiment measures temporal token redundancy.\n"
    )

    f.write(
        "It does not measure actual DINOv2 compute reduction "
        "or speedup.\n\n"
    )

    f.write(
        "SUMMARY\n"
    )

    f.write(
        "=" * 60 + "\n\n"
    )

    f.write(
        summary.to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# 5. Verify everything
# ------------------------------------------------------------

print("=" * 70)
print("STAGE 2 RESULTS SAVED")
print("=" * 70)

saved_files = [
    raw_path,
    summary_path,
    json_path,
    report_path,
]

for path in saved_files:

    print(
        ("✅" if path.exists() else "❌"),
        path
    )

print("\nResults directory:")
print(RESULTS)

print("\nDirectory contents:")

for path in sorted(RESULTS.iterdir()):
    print(
        f"  {path.name:50s}"
        f" {path.stat().st_size / 1024:.1f} KB"
    )

print("\n🎉 Stage-2 results are saved in /kaggle/working/")

STAGE 2 RESULTS SAVED
✅ /kaggle/working/woodscape_results_clean/stage2_temporal_token_raw.csv
✅ /kaggle/working/woodscape_results_clean/stage2_temporal_token_summary.csv
✅ /kaggle/working/woodscape_results_clean/stage2_temporal_token_summary.json
✅ /kaggle/working/woodscape_results_clean/stage2_report.txt

Results directory:
/kaggle/working/woodscape_results_clean

Directory contents:
  stage2_report.txt                                  2.4 KB
  stage2_temporal_token_raw.csv                      17.1 KB
  stage2_temporal_token_summary.csv                  2.0 KB
  stage2_temporal_token_summary.json                 5.9 KB

🎉 Stage-2 results are saved in /kaggle/working/


In [25]:
from pathlib import Path
import shutil

RESULTS = Path("/kaggle/working/woodscape_results_clean")
ZIP_BASE = Path("/kaggle/working/WoodScape_Stage2_Results")

assert RESULTS.exists(), f"Results folder not found: {RESULTS}"

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=RESULTS.parent,
    base_dir=RESULTS.name
)

print("=" * 70)
print("STAGE 2 RESULTS ZIPPED")
print("=" * 70)
print("ZIP:", zip_path)
print("Size:", round(Path(zip_path).stat().st_size / 1024, 2), "KB")

print("\nContents:")
with __import__("zipfile").ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(" ", name)

print("\n✅ Ready to download/upload.")

STAGE 2 RESULTS ZIPPED
ZIP: /kaggle/working/WoodScape_Stage2_Results.zip
Size: 6.41 KB

Contents:
  woodscape_results_clean/
  woodscape_results_clean/stage2_temporal_token_summary.json
  woodscape_results_clean/stage2_temporal_token_raw.csv
  woodscape_results_clean/stage2_temporal_token_summary.csv
  woodscape_results_clean/stage2_report.txt

✅ Ready to download/upload.
